# Explainability and Analyst Reason Codes

## Objective
Explain why the model classifies a transaction as risky. The dashboard converts top-feature evidence into analyst-readable reason codes.


In [ ]:
# Import the core libraries used for data analysis and visualisation.
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make notebook tables easier to inspect during review.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Add the project src folder so notebook code reuses production pipeline logic.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT / 'src'))

from fraud_pipeline import BASE_FEATURES, load_transactions, train_top_feature_model, predict_transaction


In [ ]:
# Load data and train the same top-feature model used by the Streamlit app.
transactions = load_transactions(REPO_ROOT)
bundle = train_top_feature_model(transactions, top_n=10)


## Example Transaction Explanation
The example below takes a high-risk looking transaction profile, predicts fraud probability, and explains the key drivers.


In [ ]:
# Select a real fraudulent transaction from the dataset for explanation.
# This keeps the example grounded in observed transaction behaviour.
fraud_examples = transactions[transactions['fraud_label'] == 1]
example_transaction = (
    fraud_examples.sort_values('device_risk_score', ascending=False)
    .iloc[0][BASE_FEATURES]
    .to_dict()
)

probability, label, drivers = predict_transaction(
    bundle,
    example_transaction,
    transactions,
)

print('Prediction:', label)
print('Fraud probability:', f'{probability:.2%}')
display(drivers)


In [ ]:
# Visualise the top reason codes for the sample transaction.
plot_df = drivers.sort_values('model_importance', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(plot_df['feature'], plot_df['model_importance'])
plt.title('Top Feature Contributions for Example Transaction')
plt.xlabel('Mutual-information feature importance')
plt.ylabel('Feature')
plt.show()


## Explainability Insight
The application does not only return a fraud label. It gives analysts a ranked list of reason codes, showing which transaction attributes most influenced prioritisation and how the input compares with historical behaviour.
